# 01 · Chunking experiments
Load `data/bean-scene.md`, try different `chunk_size` / `chunk_overlap` values and inspect the chunks and their metadata **before** anything is embedded. No API keys needed.

In [1]:
import pandas as pd
from bean_rag.config import settings
from bean_rag.loader import load_files
from bean_rag.splitter import split_files

pd.set_option('display.max_colwidth', 90)
files = load_files()
for name, text in files:
    print(f'{name}: {len(text):,} characters, {text.count(chr(10)):,} lines')
print('current settings -> chunk_size', settings.chunk_size, '| chunk_overlap', settings.chunk_overlap)

bean-scene.md: 19,813 characters, 349 lines
current settings -> chunk_size 800 | chunk_overlap 100


## Compare settings
Small chunks = precise search but less context per chunk. Big chunks = more context but several topics mixed together, so search gets fuzzier. *split sections* counts headings that were cut into more than one chunk.

In [2]:
rows = []
for size in (300, 500, 800, 1200):
    for overlap in (0, 100, 150):
        if overlap >= size:
            continue
        chunks = split_files(files, chunk_size=size, chunk_overlap=overlap)
        lengths = [len(c.page_content) for c in chunks]
        paths = pd.Series([c.metadata['path'] for c in chunks])
        rows.append({'chunk_size': size, 'overlap': overlap, 'chunks': len(chunks),
                     'avg_len': round(sum(lengths) / len(lengths)), 'max_len': max(lengths),
                     'split_sections': int((paths.value_counts() > 1).sum())})
pd.DataFrame(rows)

,chunk_size,overlap,chunks,avg_len,max_len,split_sections
0,300,0,98,218,335,15
1,300,100,101,224,335,15
2,300,150,103,232,335,15
3,500,0,85,247,543,5
4,500,100,85,248,543,5
5,500,150,85,252,543,5
6,800,0,80,260,654,0
7,800,100,80,260,654,0
8,800,150,80,260,654,0
9,1200,0,80,260,654,0


Headings already cut the file into topic-sized pieces, so bigger sizes never merge two topics — they only stop splitting the few long sections. At 300 or 500 some sections are cut and leave stray fragments (e.g. *"You don't need an account to order"* on its own); from 800 up **no section is split** and the result stops changing. So we use **800 / 100**: one whole section (one menu item, one FAQ) per chunk, with 100 chars of overlap ready for when a section grows past 800.

## Chunks with the chosen settings

In [3]:
chunks = split_files(files)
df = pd.DataFrame([{**c.metadata, 'chars': len(c.page_content)} for c in chunks])
print(len(chunks), 'chunks')
df[['chunk_index', 'type', 'path', 'item_name', 'price', 'chars']].head(25)

80 chunks


,chunk_index,type,path,item_name,price,chars
0,0,about,About Bean Scene > Who we are,NaN,NaN,371
1,1,about,About Bean Scene > Our story,NaN,NaN,543
2,2,about,About Bean Scene > Our mission,NaN,NaN,186
3,3,about,About Bean Scene > Founder,NaN,NaN,175
4,4,about,About Bean Scene > Bean Scene in numbers,NaN,NaN,131
5,5,about,About Bean Scene > How our coffee is made,NaN,NaN,394
6,6,about,About Bean Scene > Why we are different,NaN,NaN,296
7,7,about,About Bean Scene > Our atmosphere,NaN,NaN,203
8,8,about,About Bean Scene > What customers say,NaN,NaN,654
9,9,menu,Menu > Menu overview,NaN,NaN,380


In [4]:
df.groupby('type').agg(chunks=('chunk_index', 'count'), avg_chars=('chars', 'mean')).round()

,chunks,avg_chars
type,,
about,9,328.0
account,2,237.0
contact,6,162.0
dietary,3,395.0
faq,25,154.0
guide,8,299.0
menu,17,309.0
order,5,290.0
recommendation,3,471.0


In [5]:
df['chars'].describe().round()

count     80.0
mean     260.0
std      127.0
min       95.0
25%      164.0
50%      234.0
75%      323.0
max      654.0
Name: chars, dtype: float64

## Look at a few chunks exactly as they will be embedded

In [6]:
for c in chunks[12:15] + chunks[-2:]:
    print('─' * 80)
    print(c.metadata)
    print(c.page_content)

────────────────────────────────────────────────────────────────────────────────
{'source': 'bean-scene.md', 'section': 'Menu', 'path': 'Menu > Hot Coffee > Cappuccino', 'type': 'menu', 'chunk_index': 12, 'item_name': 'Cappuccino', 'category': 'Hot Coffee', 'price': 8.5}
Menu > Hot Coffee > Cappuccino
Category: Hot Coffee. Price: $8.50. Badge: Bestseller.
Description: coffee 50%, milk 50%. Bold espresso topped with silky steamed milk and a layer of foam — our most loved hot drink.
Taste: balanced, creamy and smooth with a strong coffee flavour. Contains milk. Served hot.
Best for: mornings and anyone who wants a classic. Pairs well with a Butter Croissant.
────────────────────────────────────────────────────────────────────────────────
{'source': 'bean-scene.md', 'section': 'Menu', 'path': 'Menu > Hot Coffee > Chai Latte', 'type': 'menu', 'chunk_index': 13, 'item_name': 'Chai Latte', 'category': 'Hot Coffee', 'price': 8.5}
Menu > Hot Coffee > Chai Latte
Category: Hot Coffee. Price: $8.

## Next
Happy with the chunks? Change `chunk_size` / `chunk_overlap` in `src/bean_rag/config.py` if needed, then run `python -m bean_rag.ingest` to store them in Supabase, and open `02_retrieval.ipynb`.